# Simulation control

Supplementary figure explaining and validating the "simulated data" control used to
compare RS/OF (running-speed / optic-flow) integration between free locomotion
(closed loop, spheres) and the motorised-wheel (treadmill) condition.

For each neuron, a simulated response is built by (1) taking the RS/OF Gaussian fit
obtained on the treadmill data and forcing it to be circular (isotropic in log(RS) and
log(OF)), then (2) predicting a dF/F trace from that circularised fit given the actual
RS/OF trajectory of a recording, and (3) convolving the predicted trace with a
biexponential calcium kernel. If the real (non-simulated) fit ellipses are more
elongated/oriented than this simulated-circular control, it indicates genuine RS/OF
integration rather than an artefact of the calcium indicator's dynamics.

This figure uses only the revision ("colasa_3d-vision_revisions") treadmill sessions
(see `v1_depth_map/revisions/revision_sessions.py`, sessions tagged `"motor"`).

See `v1_depth_map/revisions/treadmill.ipynb` and
`v1_depth_map/presentations/treadmill_20260501_poster_swc.ipynb` for the exploratory
analysis this figure is drawn from.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import matplotlib

matplotlib.rcParams["pdf.fonttype"] = 42  # for pdfs
matplotlib.rcParams["svg.fonttype"] = "none"  # for svgs

In [ ]:
import matplotlib.pyplot as plt
import flexiznam as flz
from cottage_analysis.analysis import fit_gaussian_blob as fit_gb
from cottage_analysis.analysis.spheres.simulation import make_biexponential_kernel
from cottage_analysis.pipelines import pipeline_utils
from cottage_analysis.plotting.rsof_plots import plot_RS_OF_fit, plot_RS_OF_matrix

from v1_depth_map.paths import get_figures_roots
from v1_depth_map.figure_utils import treadmill
from v1_depth_map.figure_utils.rsof_integration import add_ellipse_schematics

cm = 1 / 2.54
fontsize_dict = {"title": 7, "label": 7, "tick": 5, "legend": 5}

In [ ]:
# Set default font to Arial
import matplotlib.font_manager as fm

# optional, can be None or the path to arial.ttf:
arial_font_path = (
    "/Volumes/BlackPasspo/v1_depth_map/processed/v1_manuscript_figures/fonts/arial.ttf"
)
# "/nemo/lab/znamenskiyp/home/shared/resources/fonts/arial.ttf"
# set matplotlib options
if arial_font_path is not None:
    arial_prop = fm.FontProperties(fname=arial_font_path)
    plt.rcParams["font.family"] = arial_prop.get_name()
    plt.rcParams.update({"mathtext.default": "regular"})  # make math mode also Arial
    fm.fontManager.addfont(arial_font_path)

## Load data

Revision (colasa) project only.


In [ ]:
project = "colasa_3d-vision_revisions"
flexilims_session = flz.get_flexilims_session(project)
READ_ROOT, SAVE_ROOT = get_figures_roots(flexilims_session)
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
# Population data (all colasa treadmill/"motor" sessions), including the
# simulated-response dataframes used for the polar-scatter panels below.
TDECAY = 2
TRISE = 0.15

(
    neurons_df,
    simul_df_treadmill,
    simul_df_spheres,
    valid_sessions,
    treadmill_sessions,
) = treadmill.load_treadmill_population_neurons_df(
    flexilims_session,
    protocol_base="SpheresTubeMotor",
    tdecay=TDECAY,
    trise=TRISE,
    load_simulated=True,
)

In [ ]:
# Example session/cell used for the schematic panels (kernel, circularising fit,
# simulated trace). Same example as in the exploratory notebooks.
EXAMPLE_SESSION = "PZAG17.3a_S20250402"
EXAMPLE_CELL = "PZAG17.3a_S20250402_83"
example_mouse, example_session = EXAMPLE_SESSION.split("_")

ndf, trials_df_tm, trials_df_sphere = pipeline_utils.load_treadmill_and_sphere_datasets(
    project,
    example_mouse,
    example_session,
    photodiode_protocol=5,
    filter_datasets={"anatomical_only": 3, "annotated": True},
    recording_type="two_photon",
    protocol_base_sphere="SpheresPermTubeReward",
)

suite2p_ds = flz.get_datasets(
    origin_name=EXAMPLE_SESSION,
    dataset_type="suite2p_rois",
    filter_datasets={"annotated": True},
    flexilims_session=flexilims_session,
    allow_multiple=False,
)
fs = suite2p_ds.extra_attributes["fs"]

example_cell = neurons_df[neurons_df.roi_uid == EXAMPLE_CELL].iloc[0]
roi = example_cell.roi

## Build the circularised-fit control for the example cell

Force the treadmill RS/OF Gaussian fit to be circular (isotropic), swap in the
simulated dF/F for that ROI, and compare the resulting fit to the real one.


In [ ]:
rs_bins, of_bins, tick_dict = treadmill.compute_treadmill_rsof_bins(trials_df_tm)
range_kwargs = dict(
    log_range={"log_base": 2}, rs_bins=rs_bins, of_bins=of_bins, tick_dict=tick_dict
)

# Simulated dF/F trace for this ROI (treadmill condition)
simul_data = simul_df_treadmill[simul_df_treadmill.roi_uid == example_cell.roi_uid]
assert len(simul_data) == 1
simul_data = simul_data.iloc[0]

# `simulated_responses_fit_treadmill_*.parquet` is a precomputed artifact: its
# `fake_dff` trace was sliced into trials using whatever onset-detection `method`
# `cottage_analysis.analysis.treadmill.process_imaging_df` defaulted to at the time
# it was generated. That file predates the "plateau" onset detector (added/defaulted
# to in cottage_analysis long after this parquet's mtime), which is now `trials_df_tm`'s
# default -- so `trials_df_tm`'s trial boundaries no longer match `fake_dff`'s. Reload
# with the older `method="model"` to recover matching per-trial frame counts.
_, trials_df_tm_for_sim, _ = pipeline_utils.load_treadmill_and_sphere_datasets(
    project,
    example_mouse,
    example_session,
    photodiode_protocol=5,
    filter_datasets={"anatomical_only": 3, "annotated": True},
    recording_type="two_photon",
    protocol_base_sphere="SpheresPermTubeReward",
    tread_kwargs=dict(method="model"),
)

trials_df_tm_simul = trials_df_tm_for_sim.copy()
trials_df_tm_simul["dff_stim"] = [dff.copy() for dff in trials_df_tm_for_sim.dff_stim]
trials_df_tm_simul.dff_stim += np.nan
indices = np.hstack([0, trials_df_tm_for_sim.dff_stim.map(len).values.cumsum()])
assert indices[-1] == len(simul_data.fake_dff), (
    f"trials_df_tm_for_sim has {indices[-1]} total frames but fake_dff has "
    f"{len(simul_data.fake_dff)} -- trial boundaries no longer match the precomputed "
    "simulated-response file; re-check the `tread_kwargs` used here against however "
    "`simulate_and_fit_session` generated it."
)
for i in range(len(trials_df_tm_simul["dff_stim"])):
    dff_trial = trials_df_tm_simul.at[i, "dff_stim"]
    dff_trial[:, roi] = simul_data.fake_dff[indices[i] : indices[i + 1]]
    trials_df_tm_simul.at[i, "dff_stim"] = dff_trial

# Circularised copy of every ROI's fit (needed by plot_RS_OF_fit's sfx="_circular_sim")
popt_list = []
ndf = ndf.copy()
for popt in ndf["rsof_popt_closedloop_g2d_treadmill"].values:
    if popt is None or np.isnan(popt).any():
        popt_model = None
    else:
        # Reduce the major axis to match the minor axis -> isotropic/circular fit
        popt_model = popt.copy()
        popt_model[3] = popt_model[4] = min(popt[3:5])
    popt_list.append(popt_model)
ndf["rsof_popt_closedloop_g2d_circular_sim"] = popt_list
ndf["rsof_test_rsq_closedloop_g2d_circular_sim"] = ndf[
    "rsof_test_rsq_closedloop_g2d_treadmill"
]

## Build the same circularised-fit control for free locomotion (spheres)

Same as above, but using the closed-loop (spheres) fit and RS/OF trajectory instead
of the treadmill ones.


In [ ]:
# Simulated dF/F trace for this ROI (free-locomotion/sphere condition)
simul_data_sphere = simul_df_spheres[simul_df_spheres.roi_uid == example_cell.roi_uid]
assert len(simul_data_sphere) == 1
simul_data_sphere = simul_data_sphere.iloc[0]

# Unlike treadmill trial cropping, sphere trial cropping has no tunable
# onset-detection "method" (it is cut deterministically from photodiode-derived
# stim transitions), and `simulate_and_fit_session` reuses the same sync/trial-
# extraction defaults as `trials_df_sphere` above -- so no special reload is
# needed here to recover matching per-trial frame counts.
trials_df_sphere_simul = trials_df_sphere.copy()
trials_df_sphere_simul["dff_stim"] = [dff.copy() for dff in trials_df_sphere.dff_stim]
trials_df_sphere_simul.dff_stim += np.nan
indices_sphere = np.hstack([0, trials_df_sphere.dff_stim.map(len).values.cumsum()])
assert indices_sphere[-1] == len(simul_data_sphere.fake_dff), (
    f"trials_df_sphere has {indices_sphere[-1]} total frames but fake_dff has "
    f"{len(simul_data_sphere.fake_dff)} -- trial boundaries no longer match the "
    "precomputed simulated-response file; re-check the sphere sync/trial-extraction "
    "kwargs used here against however `simulate_and_fit_session` generated it."
)
for i in range(len(trials_df_sphere_simul["dff_stim"])):
    dff_trial = trials_df_sphere_simul.at[i, "dff_stim"]
    dff_trial[:, roi] = simul_data_sphere.fake_dff[
        indices_sphere[i] : indices_sphere[i + 1]
    ]
    trials_df_sphere_simul.at[i, "dff_stim"] = dff_trial

# Circularised copy of every ROI's free-locomotion (closed-loop, spheres) fit
# (needed by plot_RS_OF_fit's sfx="_circular_sim_free")
popt_list_free = []
for popt in ndf["rsof_popt_closedloop_g2d"].values:
    if popt is None or np.isnan(popt).any():
        popt_model = None
    else:
        # Reduce the major axis to match the minor axis -> isotropic/circular fit
        popt_model = popt.copy()
        popt_model[3] = popt_model[4] = min(popt[3:5])
    popt_list_free.append(popt_model)
ndf["rsof_popt_closedloop_g2d_circular_sim_free"] = popt_list_free
ndf["rsof_test_rsq_closedloop_g2d_circular_sim_free"] = ndf[
    "rsof_test_rsq_closedloop_g2d"
]

## Prepare the example sphere trial trace

Running speed / optic flow for a couple of example sphere trials, and the dF/F
response predicted by the example cell's (real) circularised Gaussian fit, convolved
with the biexponential calcium kernel.


In [ ]:
# Sphere trials used for the simulated-trace schematic
example_sphere_indices = np.arange(29, 31)

data_sphere = np.array([])
of_sphere = np.array([])
stim_part_sphere = np.array([])

for itrial, idx in enumerate(example_sphere_indices):
    trial_series = trials_df_sphere.iloc[idx]

    if itrial == 0:
        # Start with an initial blank period
        data_sphere = trial_series.RS_blank_pre[-20:]
        of_sphere = np.zeros_like(data_sphere)
        stim_part_sphere = np.zeros(data_sphere.shape, dtype=int)

    data_sphere = np.hstack([data_sphere, trial_series.RS_stim, trial_series.RS_blank])
    of_sphere = np.hstack(
        [of_sphere, trial_series.OF_stim, np.zeros_like(trial_series.RS_blank)]
    )

    stim_id = trial_series.name
    stim_part_sphere = np.hstack(
        [
            stim_part_sphere,
            np.ones(trial_series.RS_stim.shape) * stim_id,
            np.zeros(trial_series.RS_blank.shape),
        ]
    )

time_axis_sphere = np.arange(len(data_sphere)) / fs

popt = example_cell.rsof_popt_closedloop_g2d_treadmill
with np.errstate(divide="ignore", invalid="ignore"):
    rs_log = np.log(data_sphere)
    of_log = np.log(np.degrees(of_sphere))
# Handle log(0) frames (blank periods) - map to a very negative value
rs_log[np.isnan(rs_log) | np.isinf(rs_log)] = -10
of_log[np.isnan(of_log) | np.isinf(of_log)] = -10

pred_dff_sphere = fit_gb.gaussian_2d((rs_log, of_log), *popt, min_sigma=0.25)
# Same kernel builder as the actual simulation pipeline (previously duplicated
# locally as `treadmill.make_simulation_kernel`); peak-normalized ("max") to
# match its new default. `make_biexponential_kernel` only returns the kernel
# array (no time vector), so rebuild the matching time axis for the schematic.
kernel_norm = make_biexponential_kernel(
    tau_decay=TDECAY, tau_rise=TRISE, frame_rate=fs, normalization="max"
)
time_kernel = np.arange(len(kernel_norm)) / fs
sim_dff_sphere = np.convolve(pred_dff_sphere, kernel_norm, mode="full")[
    : len(pred_dff_sphere)
]


## Assemble the figure


In [ ]:
fig = plt.figure(figsize=(18 * cm, 18 * cm))
ax = fig.add_axes([0, 0, 1, 1])
ax.set_xticks([])
ax.set_yticks([])

# Row bands (top to bottom), all within [0, 1] so nothing bleeds outside the
# (fixed) figure canvas.
row_y = {
    "kernel_traces": (0.86, 0.11),  # (A) kernel + (C) RS/OF/dFF traces
    "treadmill_mat": (0.3, 0.15),  # (B) matrices, motorised wheel (bottom row)
    "free_mat": (0.455, 0.15),  # (B) matrices, free locomotion (top row) --
    # 0.005 above "treadmill_mat" so the two rows are almost touching.
    "polar": (0.02, 0.15),  # (D)/(E) polar scatters
}


def add_matrix_colorbar(fig, ax, vmin, vmax, fontsize_dict):
    """Thin dF/F colorbar right next to `ax` -- very close, half the axis
    height, vertically centred on it."""
    pos = ax.get_position()
    cbar_w = pos.width * 0.06
    cbar_h = pos.height * 0.5
    cax = fig.add_axes(
        [
            pos.x0 + pos.width * 1.03,
            pos.y0 + (pos.height - cbar_h) / 2,
            cbar_w,
            cbar_h,
        ]
    )
    cbar = fig.colorbar(ax.images[0], cax=cax)
    cbar.set_ticks([vmin, vmax])
    cax.tick_params(labelsize=fontsize_dict.get("legend", 10), length=2, pad=2)
    return cax


# --- (A) Exponential decay kernel schematic ---
if True:
    y0, h = row_y["kernel_traces"]
    ax_kernel = fig.add_axes([0.05, y0, 0.10, h])
    ax_kernel.plot(time_kernel, kernel_norm, color="k", lw=2)
    ax_kernel.set_title(
        f"Exponential\ndecay $\\tau$={TDECAY}s", fontsize=fontsize_dict["label"]
    )
    for spine in ax_kernel.spines.values():
        spine.set_visible(False)
    ax_kernel.set_xticks([])
    ax_kernel.set_yticks([])

# --- (C) Simulated sphere-trial trace: RS / OF / simulated dF/F ---
# Stacked to the right of the kernel schematic, within the same row band.
if True:
    x0, w = 0.28, 0.3
    y0, h = row_y["kernel_traces"]
    gap = 0.01
    h_rs, h_of, h_dff = 0.4 * (h - 2 * gap), 0.2 * (h - 2 * gap), 0.4 * (h - 2 * gap)
    top = y0 + h
    ax_rs = fig.add_axes([x0, top - h_rs, w, h_rs])
    ax_of = fig.add_axes([x0, top - h_rs - gap - h_of, w, h_of])
    ax_dff = fig.add_axes([x0, y0, w, h_dff])

    ax_rs.plot(time_axis_sphere, data_sphere * 100, color="black", linewidth=1.5)
    ax_rs.set_ylabel("RS", fontsize=fontsize_dict["label"])

    ax_of.plot(time_axis_sphere, np.degrees(of_sphere), color="red", linewidth=1.5)
    ax_of.set_ylabel("OF", fontsize=fontsize_dict["label"])

    ax_dff.plot(time_axis_sphere, sim_dff_sphere, color="blue", linewidth=2)
    ax_dff.set_ylabel(r"$\Delta$F/F", fontsize=fontsize_dict["label"])

    for trace_ax in [ax_rs, ax_of, ax_dff]:
        for trial_id in np.unique(stim_part_sphere[stim_part_sphere > 0]):
            mask = stim_part_sphere == trial_id
            start_time, end_time = time_axis_sphere[mask][0], time_axis_sphere[mask][-1]
            trace_ax.axvspan(start_time, end_time, color="gray", alpha=0.1)
        for spine in trace_ax.spines.values():
            spine.set_visible(False)
        trace_ax.set_xticks([])
        trace_ax.set_yticks([])

# --- (B) data, real fit, circularised (simulated-circular) fit, and
# simulated data, RS/OF matrices. Top row: free locomotion (spheres), column
# titles sit above this row. Bottom row: motorised wheel (treadmill) -- the
# only row that keeps the running-speed x-axis label/tick labels. The two
# rows are packed almost touching vertically (see `row_y` above); columns are
# packed close horizontally (small `gap` below).
col_w = 0.12
gap = 0.0005
col_x = {
    "data": 0.08,
    "real": 0.08 + (col_w + gap),
    "circ": 0.08 + 2 * (col_w + gap),
    "sim": 0.08 + 3 * (col_w + gap),
}
vmin, vmax = 0, 0.8

# --- (B, leftmost, bottom row) data: actual binned RS/OF matrix built from
# the real (non-simulated) dF/F trace -- the ground truth the "Fit" column is
# fit to, and the point of comparison for the "Simulated data" column further
# right. The "data" and "sim" matrix panels of a given row share one colour
# scale (max of their individual vmin/vmax, set below), separate from the
# "Fit"/"Circularised fit" scale above -- raw dF/F amplitude lives on a very
# different scale from the normalised parametric-fit amplitude.
if True:
    y0, h = row_y["treadmill_mat"]
    ax_mat_real = fig.add_axes([col_x["data"], y0, col_w, h])
    vmin_real, vmax_real = plot_RS_OF_matrix(
        trials_df=trials_df_tm,
        roi=roi,
        is_closed_loop=1,
        ax=ax_mat_real,
        cbar_width=None,
        fontsize_dict=fontsize_dict,
        **range_kwargs,
    )
    ax_mat_real.set_xlabel("")

if True:
    y0, h = row_y["treadmill_mat"]
    ax_fit_real = fig.add_axes([col_x["real"], y0, col_w, h])
    plot_RS_OF_fit(
        neurons_df=ndf,
        roi=roi,
        model="g2d",
        sfx="_treadmill",
        ax=ax_fit_real,
        cbar_width=None,
        label_r2=False,
        vmin=vmin,
        vmax=vmax,
        fontsize_dict=fontsize_dict,
        **range_kwargs,
    )
    ax_fit_real.set_ylabel("")
    ax_fit_real.set_yticklabels([])
    ax_fit_real.set_xlabel("")
if True:
    y0, h = row_y["treadmill_mat"]
    ax_fit_circ = fig.add_axes([col_x["circ"], y0, col_w, h])
    plot_RS_OF_fit(
        neurons_df=ndf,
        roi=roi,
        model="g2d",
        sfx="_circular_sim",
        ax=ax_fit_circ,
        cbar_width=None,
        label_r2=False,
        vmin=vmin,
        vmax=vmax,
        fontsize_dict=fontsize_dict,
        **range_kwargs,
    )
    ax_fit_circ.set_ylabel("")
    ax_fit_circ.set_yticklabels([])
    ax_fit_circ.set_xlabel("")

# --- (B') Simulated data: actual binned RS/OF matrix built from the fake_dff
# trace (i.e. the circularised fit convolved with the calcium kernel and run
# through the real RS/OF trajectory), constructed exactly like a real-data
# RS/OF matrix would be -- should resemble the circularised fit if the
# simulation pipeline faithfully reproduces the fit's shape.
y0, h = row_y["treadmill_mat"]
ax_mat_sim = fig.add_axes([col_x["sim"], y0, col_w, h])
vmin_sim, vmax_sim = plot_RS_OF_matrix(
    trials_df=trials_df_tm_simul,
    roi=roi,
    is_closed_loop=1,
    ax=ax_mat_sim,
    cbar_width=None,
    fontsize_dict=fontsize_dict,
    **range_kwargs,
)
ax_mat_sim.set_ylabel("")
ax_mat_sim.set_yticklabels([])

# Unify the "data"/"sim" colour scale for this row and add one colorbar next
# to the last (rightmost, "Simulated data") axis of the row.
row_vmin_tm, row_vmax_tm = 0, 0.8
for mat_ax in (ax_mat_real, ax_mat_sim):
    mat_ax.images[0].set_clim(row_vmin_tm, row_vmax_tm)
add_matrix_colorbar(fig, ax_mat_sim, row_vmin_tm, row_vmax_tm, fontsize_dict)

# --- (B, top row) Same four panels, for free locomotion (spheres). Column
# titles live here; the x-axis (running-speed label/tick labels) is stripped
# from every panel in this row since the bottom (treadmill) row already
# carries it and the two rows are almost touching.
y0, h = row_y["free_mat"]
ax_mat_real_free = fig.add_axes([col_x["data"], y0, col_w, h])
vmin_real_free, vmax_real_free = plot_RS_OF_matrix(
    trials_df=trials_df_sphere,
    roi=roi,
    is_closed_loop=1,
    ax=ax_mat_real_free,
    cbar_width=None,
    fontsize_dict=fontsize_dict,
    **range_kwargs,
)
ax_mat_real_free.set_xlabel("")
ax_mat_real_free.set_xticklabels([])
ax_mat_real_free.set_title("Data", fontsize=fontsize_dict["label"])

ax_fit_real_free = fig.add_axes([col_x["real"], y0, col_w, h])
plot_RS_OF_fit(
    neurons_df=ndf,
    roi=roi,
    model="g2d",
    sfx="",
    ax=ax_fit_real_free,
    cbar_width=None,
    label_r2=False,
    vmin=vmin,
    vmax=vmax,
    fontsize_dict=fontsize_dict,
    **range_kwargs,
)
ax_fit_real_free.set_ylabel("")
ax_fit_real_free.set_yticklabels([])
ax_fit_real_free.set_xlabel("")
ax_fit_real_free.set_xticklabels([])
ax_fit_real_free.set_title("Fit", fontsize=fontsize_dict["label"])

ax_fit_circ_free = fig.add_axes([col_x["circ"], y0, col_w, h])
plot_RS_OF_fit(
    neurons_df=ndf,
    roi=roi,
    model="g2d",
    sfx="_circular_sim_free",
    ax=ax_fit_circ_free,
    cbar_width=None,
    label_r2=False,
    vmin=vmin,
    vmax=vmax,
    fontsize_dict=fontsize_dict,
    **range_kwargs,
)
ax_fit_circ_free.set_ylabel("")
ax_fit_circ_free.set_yticklabels([])
ax_fit_circ_free.set_xlabel("")
ax_fit_circ_free.set_xticklabels([])
ax_fit_circ_free.set_title("Circularised fit", fontsize=fontsize_dict["label"])

ax_mat_sim_free = fig.add_axes([col_x["sim"], y0, col_w, h])
vmin_sim_free, vmax_sim_free = plot_RS_OF_matrix(
    trials_df=trials_df_sphere_simul,
    roi=roi,
    is_closed_loop=1,
    ax=ax_mat_sim_free,
    cbar_width=None,
    fontsize_dict=fontsize_dict,
    **range_kwargs,
)
ax_mat_sim_free.set_ylabel("")
ax_mat_sim_free.set_yticklabels([])
ax_mat_sim_free.set_xlabel("")
ax_mat_sim_free.set_xticklabels([])
ax_mat_sim_free.set_title("Simulated data", fontsize=fontsize_dict["label"])

# Unify the "data"/"sim" colour scale for this row and add one colorbar next
# to the last (rightmost, "Simulated data") axis of the row.
row_vmin_free = 0
row_vmax_free = 0.8
for mat_ax in (ax_mat_real_free, ax_mat_sim_free):
    mat_ax.images[0].set_clim(row_vmin_free, row_vmax_free)
add_matrix_colorbar(fig, ax_mat_sim_free, row_vmin_free, row_vmax_free, fontsize_dict)

# Row labels, to the left of the "Data" column
fig.text(
    0.03,
    row_y["treadmill_mat"][0] + row_y["treadmill_mat"][1] / 2,
    "Motorized wheel",
    rotation=90,
    ha="center",
    va="center",
    fontsize=fontsize_dict["label"],
)
fig.text(
    0.03,
    row_y["free_mat"][0] + row_y["free_mat"][1] / 2,
    "Free locomotion",
    rotation=90,
    ha="center",
    va="center",
    fontsize=fontsize_dict["label"],
)

# --- (D)/(E) Polar eccentricity/angle scatter, simulated responses ---
if True:
    y0, h = row_y["polar"]
    ax_free = fig.add_axes([0.06, y0, 0.42, h], projection="polar")
    ax_motor = fig.add_axes([0.54, y0, 0.42, h], projection="polar")

    for pax, simul_df, sfx, title, color in zip(
        [ax_free, ax_motor],
        [simul_df_spheres, simul_df_treadmill],
        ["", "_treadmill"],
        ["Free locomotion", "Motorized wheel"],
        ["orange", "dodgerblue"],
    ):
        ndf_pop = neurons_df[neurons_df[f"rsof_neuron{sfx}"]]
        ndf_sim = simul_df[simul_df.roi_uid.isin(ndf_pop.roi_uid)]
        pax.scatter(
            np.radians(ndf_sim[f"g2d_theta{sfx}"]),
            ndf_sim[f"g2d_eccentricity{sfx}"],
            alpha=0.4,
            s=10,
            c=color,
            edgecolor="k",
            linewidths=0.3,
            clip_on=False,
        )
        pax.set_theta_zero_location("E")  # 0 is East (Right) -> OF
        pax.set_thetalim(np.radians(-45), np.radians(135))
        pax.set_xticks(np.radians([-45, 0, 45, 90, 135]))
        pax.set_xticklabels(
            ["-45°", "0°", "45°", "90°", "135°"], fontsize=fontsize_dict["tick"]
        )
        pax.tick_params(pad=0)
        pax.text(
            np.radians(-62),
            0.5,
            "Eccentricity",
            rotation=-45,
            ha="center",
            va="center",
            fontsize=fontsize_dict["label"],
        )
        pax.set_rlim(0, 1)
        pax.set_rlabel_position(0)
        pax.set_title(title, fontsize=fontsize_dict["title"])
        add_ellipse_schematics(pax, scale=1.0)

fig.savefig(
    SAVE_ROOT / "fig_supp_simulation_control.svg",
    bbox_inches="tight",
    transparent=True,
)
print(f"Saved figure to {SAVE_ROOT / 'fig_supp_simulation_control.svg'}")


In [ ]:
import numpy as np

for sfx in ["_treadmill", "_circular_sim"]:
    popt = ndf.loc[ndf.roi == roi, f"rsof_popt_closedloop_g2d{sfx}"].iloc[0]
    log_amplitude, x0, y0, log_sigma_x2, log_sigma_y2, theta, offset = popt[:7]
    amplitude = np.exp(log_amplitude)
    print(f"{sfx}: amplitude={amplitude:.3f}, offset={offset:.3f}, "
          f"peak (amplitude+offset)={amplitude + offset:.3f}")

print(f"data/sim row scale (treadmill): vmin={row_vmin_tm:.3f}, vmax={row_vmax_tm:.3f}")